# Sales Forecasting & Analytics — Exploratory Data Analysis

This notebook explores the cleaned sales dataset and previews the
feature-engineering / forecasting pipeline used by the API. Run it after
seeding the project (`python -m scripts.seed`) from the project root.

All figures are computed from the data — no conclusions are hard-coded.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))  # allow `import app...` from notebooks/

import pandas as pd
import matplotlib.pyplot as plt

from app.core.config import get_settings
from app.services.data_service import DataService

settings = get_settings()
pd.set_option('display.max_columns', None)

## 1. Load (or generate) and clean the data

In [ ]:
svc = DataService()
raw_path = settings.raw_data_path
if not os.path.exists(os.path.join('..', raw_path)) and not os.path.exists(raw_path):
    svc.generate_sample_dataset(n_records=8000)

raw = svc.load_csv()
df = svc.clean(raw)
print(f'Cleaned rows: {len(df):,}')
df.head()

## 2. Headline KPIs

In [ ]:
total_revenue = df['revenue'].sum()
total_orders = len(df)
total_quantity = df['quantity'].sum()
aov = total_revenue / total_orders
print(f'Total revenue     : {total_revenue:,.2f}')
print(f'Total orders      : {total_orders:,}')
print(f'Total quantity    : {total_quantity:,}')
print(f'Avg order value   : {aov:,.2f}')

## 3. Monthly sales trend

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])
monthly = df.groupby(df['order_date'].dt.to_period('M'))['revenue'].sum()
ax = monthly.plot(kind='line', marker='o', figsize=(11, 4), title='Monthly Sales Trend')
ax.set_ylabel('Revenue'); plt.tight_layout(); plt.show()

## 4. Category-, product- and region-wise sales

In [ ]:
by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False)
by_region = df.groupby('region')['revenue'].sum().sort_values(ascending=False)
by_product = df.groupby('product_name')['revenue'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
by_category.plot(kind='bar', ax=axes[0], title='Category-wise Sales')
by_region.plot(kind='bar', ax=axes[1], title='Region-wise Sales', color='tab:green')
plt.tight_layout(); plt.show()
by_product

## 5. Feature engineering preview (leakage-safe)

We aggregate to a daily series and build calendar + lag + rolling features.
Rolling means are computed on the **lag-1** series so the current day never
leaks into its own feature.

In [ ]:
from app.ml.preprocessing import aggregate_daily_sales
from app.ml.feature_engineering import build_features, FEATURE_COLUMNS

daily = aggregate_daily_sales(df)
feats = build_features(daily)
print('Feature columns:', list(FEATURE_COLUMNS))
feats[['date', 'sales', 'sales_lag_1', 'rolling_mean_7']].head(10)

## 6. Train a model and forecast

Uses the same `ModelTrainer` and `Forecaster` the API uses. The split is
chronological (no shuffling).

In [ ]:
from app.ml.train import ModelTrainer
from app.ml.predict import Forecaster

result, _ = ModelTrainer().train(df, model_type='auto')
print(f'Selected model: {result.model_name}')
print(f'RMSE={result.rmse:.2f}  MAE={result.mae:.2f}  R2={result.r2_score:.3f}')

forecast = Forecaster().forecast(days=14)
pd.DataFrame(forecast)